# Matriz de modelagem — Base 3

Random Forest e PLS compartilham esta matriz, não as 671 colunas de `features_base3.csv`.

Para prever PM2.5(t+1), a origem é a hora t. Entram as medições conhecidas em t, inclusive o próprio PM2.5. O calendário é o da hora t+1, porque a hora e o dia da semana já são conhecidos. A meteorologia de Guanyuan fica de fora: DEWP, PRES, RAIN e WSPM repetem Aotizhongxin.

O código que monta a matriz é `02_modelos/protocolo_final.py`. Este notebook chama a mesma função e confere o alinhamento temporal.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

current = Path.cwd()
while current != current.parent and not (current / "bases" / "grupo3").exists():
    current = current.parent
ROOT = current
sys.path.insert(0, str(ROOT / "analyses" / "grupo3" / "02_modelos"))
import protocolo_final as protocolo

frame, pm, _ = protocolo.load_frame(ROOT)
protocolo.write_protocol(ROOT, frame)
check = frame[["timestamp", "pm25_t", "y"]].copy()
check["pm25_na_origem"] = check["timestamp"].map(pm)
check["pm25_na_hora_alvo"] = (check["timestamp"] + pd.Timedelta(hours=1)).map(pm)
print("linhas", len(frame))
print(frame["split"].value_counts().to_string())
print("pm25_t igual ao PM2.5 da origem", (check["pm25_t"] - check["pm25_na_origem"]).abs().max() == 0)
print("y igual ao PM2.5 de t+1", (check["y"] - check["pm25_na_hora_alvo"]).abs().max() == 0)
print("Guanyuan ausente", not any("Guanyuan" in column for column in frame.columns))
print(pd.read_csv(ROOT / "analyses/grupo3/01_dados/outputs/dicionario_modelagem_base3.csv").to_string(index=False))


linhas 30062
split
train_interno        19494
test_final            6023
validacao_interna     4545
pm25_t igual ao PM2.5 da origem True
y igual ao PM2.5 de t+1 True
Guanyuan ausente True
          feature        disponivel_em
           pm25_t             origem t
         pm25_t_1             origem t
         pm25_t_2             origem t
        pm25_t_23             origem t
 pm25_roll_mean_3             origem t
pm25_roll_mean_24             origem t
       local_PM10             origem t
         local_CO             origem t
       local_TEMP             origem t
       local_WSPM             origem t
       local_DEWP             origem t
  Changping_PM2.5             origem t
     Wanliu_PM2.5             origem t
  target_hour_sin t+1, conhecido antes
  target_hour_cos t+1, conhecido antes
   target_dow_sin t+1, conhecido antes
   target_dow_cos t+1, conhecido antes


## Checksum

Os arquivos congelados em `bases/grupo3` têm SHA-256 em `bases/grupo3/SHA256SUMS.txt`. O hash identifica o arquivo usado nesta base; não altera as linhas.

In [2]:
checksums = (ROOT / "bases" / "grupo3" / "SHA256SUMS.txt").read_text(encoding="utf-8")
print(checksums)
payload = []
for line in checksums.splitlines():
    digest, filename = line.split(maxsplit=1)
    payload.append({"arquivo": filename, "sha256": digest})
destination = ROOT / "analyses" / "grupo3" / "01_dados" / "outputs" / "checksums_base3.json"
destination.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination)


a94fbcfc71708b6ffdc033360163d91efb7f675e496f6b5860929aa96b96351b  bases/grupo3/PRSA_Data_Aotizhongxin_20130301-20170228.csv
a2e21402a6d95fb7043f87959e6e3dcbecb957aac356a6f8028a85b047c772e7  bases/grupo3/externas/PRSA_Data_Changping_20130301-20170228.csv
59390fee3b032597b387491f5230e7edb90a8558e30217ebd6d9befbe669fbf9  bases/grupo3/externas/PRSA_Data_Guanyuan_20130301-20170228.csv
0b1a6d754a07f0def2f55434ddfa3870dcd0063b7f766a6f66b6c11bb05a377a  bases/grupo3/externas/PRSA_Data_Wanliu_20130301-20170228.csv

/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/analyses/grupo3/01_dados/outputs/checksums_base3.json
